# A3 — Character theory from scratch: S₃ and S₄

**What this notebook claims** (state without looking at phase exit):

1. **Schur orthogonality**: $\frac{1}{|G|}\sum_g \overline{\chi_i(g)}\chi_j(g) = \delta_{ij}$ for irreducible characters; characters of irreps are orthonormal class functions.
2. **Class sums** $K_r = \sum_{g \in C_r} g$ live in the **center** of the group algebra; in an irrep $\rho_i$, $\rho_i(K_r) = \lambda_r^i I$ with $\lambda_r^i = \dfrac{|C_r|\,\chi_i(C_r)}{d_i}$.
3. The $\lambda^i$ are simultaneous eigenvectors of the multiplication matrices $M_r$ with $(M_r)_{st} = c_{rst}$ (structure constants of $K_rK_s = \sum_t c_{rst}K_t$). Finding them gives the full character table **without ever writing down a single representation matrix**.
4. $\sum_i d_i^2 = |G|$ — checked as an independent consistency condition.
5. Physics: the character table *is* the symmetry bookkeeping of quantum mechanics — multiplicities of degenerate multiplets, selection rules, decompositions.

Source map: Woit ch. 2 (character theory), Zee part I. *Verify section numbers against your copy.*

## The method, in three lines

1. Compute conjugacy classes $C_r$ and structure constants $c_{rst} = \#\{(a,b)\in C_r\times C_s : ab = g_t\}$ (independent of the choice of $g_t\in C_t$).
2. A generic center element $\sum_r \alpha_r M_r$ is diagonalised; its 1-dim eigenspaces are the common eigenspaces of all $M_r$, giving $\lambda_r^i$ up to scale.
3. Scale fixed by $d_i^2 = |G|/\sum_r (\lambda_r^i)^2/|C_r|$; then $\chi_i(C_r) = \lambda_r^i d_i / |C_r|$.
Everything below is computed from these definitions only.

In [ ]:
import sympy as sp
from itertools import permutations

def compose(p, q, n):
    return tuple(p[q[i]] for i in range(n))
def inverse(p, n):
    inv = [0]*n
    for i, j in enumerate(p):
        inv[j] = i
    return tuple(inv)

def group_Sn(n):
    G = list(permutations(range(n)))
    e = tuple(range(n))
    # conjugacy classes
    seen, classes = set(), []
    for g in G:
        if g in seen:
            continue
        cls = frozenset(compose(compose(h, g, n), inverse(h, n), n) for h in G)
        seen |= cls
        classes.append(sorted(cls))
    classes.sort(key=lambda c: (len(c), c[0]))
    class_of = {}
    for r, c in enumerate(classes):
        for g in c:
            class_of[g] = r
    return G, e, classes, class_of

def structure_constants(G, classes, n):
    # c[r][s][t] = # {(a,b) in C_r x C_s : ab = g_t} for a FIXED g_t in C_t.
    # Counts are uniform across each class, so: count all products landing in C_t,
    # then divide by |C_t|. (The raw landing count is |C_t| * c[r][s][t].)
    k = len(classes)
    c = [[[0]*k for _ in range(k)] for _ in range(k)]
    for r in range(k):
        for s in range(k):
            counts = {}
            for a in classes[r]:
                for b in classes[s]:
                    ab = compose(a, b, n)
                    t = next(t for t in range(k) if ab in classes[t])
                    counts[t] = counts.get(t, 0) + 1
            for t, m in counts.items():
                assert m % len(classes[t]) == 0, "counts not uniform across class -- method broken"
                c[r][s][t] = m // len(classes[t])
    return c

def character_table(G, e, classes, class_of, n, alphas):
    # Simultaneous eigenvectors of the class-sum multiplication matrices.
    k = len(classes)
    c = structure_constants(G, classes, n)
    Ms = [sp.Matrix(k, k, lambda s, t, r=r: c[r][s][t]) for r in range(k)]
    C = sp.zeros(k)
    for r in range(k):
        C += alphas[r] * Ms[r]
    vecs = []
    for val, mult, basis in C.eigenvects():
        assert mult == 1, f"degenerate eigenspace (eigenvalue {val}); pick different alphas"
        vecs.append(basis[0])
    assert len(vecs) == k, "expected k one-dimensional eigenspaces"
    table, dims = [], []
    for v in vecs:
        lam = []
        for r in range(k):
            s0 = next(s for s in range(k) if v[s] != 0)
            ratio = sp.simplify((Ms[r] * v)[s0] / v[s0])
            assert all(sp.simplify((Ms[r] * v)[s] - ratio * v[s]) == 0 for s in range(k))
            lam.append(sp.nsimplify(ratio))
        d2 = sp.nsimplify(sp.Rational(len(G), 1) / sum(lam[r]**2 / len(classes[r]) for r in range(k)))
        d = sp.sqrt(d2)
        assert d.is_integer, f"non-integer dimension {d}"
        dims.append(int(d))
        table.append([sp.nsimplify(lam[r] * d / len(classes[r])) for r in range(k)])
    return table, dims, classes

## S₃ from scratch, and an independent check

S₃ classes (as ordered by the code, by size): $C_0 = \{e\}$, $C_1$ = 2 three-cycles, $C_2$ = 3 transpositions.
As an **independent** cross-check (not an input), we compute the three irreducible characters directly from matrices — trivial (all ones), sign (determinant of the permutation matrix), standard (fixed points minus one) — and require the scratch-computed table to match.

In [ ]:
G3, e3, cls3, cof3 = group_Sn(3)
table3, dims3, _ = character_table(G3, e3, cls3, cof3, 3, alphas=(1, 2, 4))
print("S3 classes:", [len(c) for c in cls3], " dims:", dims3)
for row in table3:
    print("  ", row)

# independent check: trivial / sign / standard computed directly from matrices
perm_matrix = lambda g: sp.Matrix(3, 3, lambda i, j: 1 if g[j] == i else 0)
direct = sorted([
    tuple(1 for _ in G3),                                            # trivial
    tuple(sp.sign(sp.det(perm_matrix(g))) for g in G3),              # sign
    tuple(sum(1 for i in range(3) if g[i] == i) - 1 for g in G3),    # standard
])
computed = sorted(tuple(row[cof3[g]] for g in G3) for row in table3)
assert computed == direct, "scratch table disagrees with direct computation!"
assert sorted(dims3) == [1, 1, 2] and sum(d*d for d in dims3) == 6
print("matches direct computation; sum d_i^2 = 6 = |S3|  -> Claims 1-4 verified for S3")

## S₄ from scratch

S₄ classes (as ordered by the code, by size): $1^4$ (1), $2^2$ (3), $2\,1^2$ (6), $4$ (6), $3\,1$ (8).
Checks: row orthogonality with class weighting, $\sum d_i^2 = 24$, and the four directly-computable rows (trivial, sign, standard = fixed-points$-$1, sign⊗standard) must appear.

In [ ]:
G4, e4, cls4, cof4 = group_Sn(4)
table4, dims4, _ = character_table(G4, e4, cls4, cof4, 4, alphas=(1, 2, 4, 8, 16))
print("S4 classes (sizes):", [len(c) for c in cls4], " dims:", sorted(dims4))
for row in table4:
    print("  ", row)

# Claim 1: row orthogonality with class weighting
for i in range(len(table4)):
    for j in range(len(table4)):
        ip = sp.Rational(sum(len(cls4[r]) * table4[i][r] * table4[j][r] for r in range(5)), len(G4))
        assert ip == (1 if i == j else 0), f"orthogonality fails at ({i},{j})"
# Claim 4: sum of squares of dims
assert sum(d*d for d in dims4) == 24 == len(G4)

# direct rows as an independent witness
def fixed(g): return sum(1 for i in range(4) if g[i] == i)
sign4 = {g: sp.sign(sp.det(sp.Matrix(4, 4, lambda i, j: 1 if g[j] == i else 0))) for g in G4}
std4 = {g: fixed(g) - 1 for g in G4}
witnesses = [
    {g: 1 for g in G4},
    sign4,
    std4,
    {g: sign4[g] * std4[g] for g in G4},
]
rows_as_chars = [tuple(row[cof4[g]] for g in G4) for row in table4]
for w in witnesses:
    assert tuple(w[g] for g in G4) in rows_as_chars, "directly computable row missing!"
print("orthogonality OK; sum d_i^2 = 24 = |S4|; all directly computable rows present")
print()
print("S4 character table (rows = irreps, cols = classes in code order 1^4, 2^2, 2 1^2, 4, 3 1):")
hdr = ["d"] + [str(len(c)) for c in cls4]
print("      " + "  ".join(f"{h:>3}" for h in hdr))
for d, row in zip(dims4, table4):
    print(f"  d={d} |" + "  ".join(f"{str(r):>3}" for r in row))

## Physics deposit for Phase A

**Why energy levels come in multiplets.** If the Hamiltonian commutes with a group $G$ of symmetries, its eigenspaces carry representations of $G$; an irreducible subspace of dimension $d$ is a multiplet of $d$ degenerate states. The character table encodes *which* multiplets can occur and how products of states decompose (Clebsch–Gordan preview) — this is Wigner-style symmetry bookkeeping, and it is the same machinery that will label particles by Poincaré irreps in Phase D.
Write the precise statements (from Woit ch. 2 / Zee part I) into *My summary*.

## My summary

*(State Schur orthogonality, the class-sum eigenvalue method, and the multiplet argument without notes.)*

-


## Gaps

- 